In [2]:
from ultralytics import YOLO
import cv2
import csv
import time

In [ ]:
from ultralytics import YOLO
import cv2
import csv
import time

model = YOLO("yolo26s-pose.pt")

cap = cv2.VideoCapture(0)

BODY_PARTS = [
    "Nose", "Left Eye", "Right Eye", "Left Ear", "Right Ear",
    "Left Shoulder", "Right Shoulder",
    "Left Elbow", "Right Elbow",
    "Left Wrist", "Right Wrist",
    "Left Hip", "Right Hip",
    "Left Knee", "Right Knee",
    "Left Ankle", "Right Ankle"
]

# Keep only shoulders and elbows
keep_indices = [5, 6, 7, 8]

file = open("data/shoulder_exercise_1/seq5.csv", "w", newline="")
writer = csv.writer(file)

# CSV Header
header = ["frame"]
for i in keep_indices:
    part = BODY_PARTS[i]
    header += [f"{part}_x", f"{part}_y"]

writer.writerow(header)

frame_id = 0
height = 1080
width = 1920

# ----------------------------
# Timers
# ----------------------------
PREP_TIME = 10       # 10-second preparation
RECORD_TIME = 20     # 20-second recording

start_time = time.time()

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Mirror like iPhone front camera
        frame = cv2.flip(frame, 1)

        results = model(frame)
        annotated = results[0].plot()

        elapsed = time.time() - start_time

        # ======================================
        # PREPARATION COUNTDOWN
        # ======================================
        if elapsed < PREP_TIME:

            remaining = PREP_TIME - elapsed

            cv2.putText(
                annotated,
                f"Get Ready: {int(remaining) + 1}s",
                (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2,
            )

        # ======================================
        # RECORDING
        # ======================================
        elif elapsed < PREP_TIME + RECORD_TIME:

            recording_elapsed = elapsed - PREP_TIME
            recording_remaining = RECORD_TIME - recording_elapsed

            cv2.putText(
                annotated,
                f"Recording: {int(recording_remaining) + 1}s",
                (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2,
            )

            keypoints = results[0].keypoints

            if keypoints is not None:
                xy = keypoints.xy

                for person_idx in range(len(xy)):
                    row = [frame_id]

                    for i in keep_indices:
                        x = xy[person_idx][i][0].item() / width
                        y = xy[person_idx][i][1].item() / height

                        row += [x, y]

                    writer.writerow(row)

                file.flush()

            frame_id += 1

        # ======================================
        # FINISH RECORDING
        # ======================================
        else:

            cv2.putText(
                annotated,
                "Recording Complete!",
                (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255, 0, 0),
                2,
            )

            cv2.imshow("YOLO Pose", annotated)
            cv2.waitKey(2000)  # Display completion message for 2 seconds
            break

        cv2.imshow("YOLO Pose", annotated)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    file.close()
    cv2.destroyAllWindows()



0: 384x640 (no detections), 99.1ms
Speed: 3.6ms preprocess, 99.1ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 85.6ms
Speed: 1.6ms preprocess, 85.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 69.9ms
Speed: 1.2ms preprocess, 69.9ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 69.3ms
Speed: 1.2ms preprocess, 69.3ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 71.3ms
Speed: 1.3ms preprocess, 71.3ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 74.9ms
Speed: 1.2ms preprocess, 74.9ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 67.2ms
Speed: 1.2ms preprocess, 67.2ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 60.4ms
Speed: 1.2ms preprocess, 60.4ms inference, 0.1ms postprocess per image at shape (

: 

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.nn.functional as F
import torch

In [30]:
df = pd.read_csv('data/detections.csv')

df.head()

,frame,Left Shoulder_x,Left Shoulder_y,Left Shoulder_conf,Right Shoulder_x,Right Shoulder_y,Right Shoulder_conf,Left Elbow_x,Left Elbow_y,Left Elbow_conf,...,Left Knee_conf,Right Knee_x,Right Knee_y,Right Knee_conf,Left Ankle_x,Left Ankle_y,Left Ankle_conf,Right Ankle_x,Right Ankle_y,Right Ankle_conf
0,0,1424.110962,993.762634,0.905596,704.344666,1020.448792,0.913182,1518.191406,1071.431396,0.005509,...,0.001585,838.337830,1080.0,0.001660,1192.874146,1080.0,0.000352,940.914551,1080.0,0.000696
1,1,1438.859985,1014.715759,0.933951,697.710754,1028.207520,0.940646,1517.736938,1080.000000,0.002958,...,0.001362,828.336609,1080.0,0.001547,1180.128418,1080.0,0.000240,928.343689,1080.0,0.000516
2,2,1446.010254,1031.935547,0.923083,698.586304,1031.863037,0.936629,1518.396851,1080.000000,0.002013,...,0.001350,817.569214,1080.0,0.001726,1171.809814,1080.0,0.000203,922.086548,1080.0,0.000456
3,3,1439.775635,1035.010986,0.920916,702.842834,1031.151489,0.937211,1517.457397,1080.000000,0.001792,...,0.001236,817.365417,1080.0,0.001657,1167.926147,1080.0,0.000184,924.311829,1080.0,0.000417
4,4,1439.810303,1035.710205,0.920352,700.170227,1030.109497,0.935427,1519.770630,1080.000000,0.001756,...,0.001230,815.877380,1080.0,0.001672,1170.679321,1080.0,0.000182,924.766174,1080.0,0.000422


In [27]:
df.describe()

,frame,Left Shoulder_x,Left Shoulder_y,Left Shoulder_conf,Right Shoulder_x,Right Shoulder_y,Right Shoulder_conf,Left Elbow_x,Left Elbow_y,Left Elbow_conf,...,Left Knee_conf,Right Knee_x,Right Knee_y,Right Knee_conf,Left Ankle_x,Left Ankle_y,Left Ankle_conf,Right Ankle_x,Right Ankle_y,Right Ankle_conf
count,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,...,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000
mean,31.246575,1479.845210,996.943710,0.769920,812.329511,1019.888796,0.806980,1607.940868,1047.239123,0.010139,...,0.001645,950.307367,1044.377972,0.001834,1299.571025,1057.897777,0.000889,1050.509838,1059.338103,0.001494
std,16.641465,109.220648,45.611458,0.263498,314.004251,38.465325,0.268999,113.336366,43.296435,0.019162,...,0.000809,239.105332,50.849688,0.001673,176.121766,34.347196,0.001602,230.457615,33.501083,0.003552
min,0.000000,1386.805908,788.172791,0.002668,690.666321,807.235657,0.004874,1470.353638,851.452148,0.001042,...,0.000724,815.877380,890.705933,0.000454,1167.926147,957.878723,0.000151,921.878235,953.080811,0.000327
25%,18.000000,1429.003174,980.209961,0.800902,702.842834,1024.550903,0.831938,1511.337036,1028.748779,0.001792,...,0.001135,835.933655,1022.260315,0.001100,1181.574951,1042.947144,0.000240,933.222900,1053.443481,0.000438
50%,32.000000,1439.810303,1007.847717,0.859499,712.725769,1030.075195,0.907504,1530.315918,1065.807129,0.003653,...,0.001362,863.668945,1080.000000,0.001483,1206.568359,1080.000000,0.000388,950.536621,1080.000000,0.000673
75%,45.000000,1473.755615,1026.478271,0.905596,721.202454,1033.263916,0.938942,1697.711060,1080.000000,0.010990,...,0.001947,926.695740,1080.000000,0.001672,1378.994873,1080.000000,0.001278,1069.377319,1080.000000,0.001288
max,59.000000,1866.227539,1050.137329,0.988900,1858.493042,1040.435791,0.962918,1876.223267,1080.000000,0.146377,...,0.004756,1857.587891,1080.000000,0.010352,1912.609131,1080.000000,0.012873,1899.409058,1080.000000,0.029463


In [28]:
df.shape

(73, 37)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 37 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   frame                73 non-null     int64  
 1   Left Shoulder_x      73 non-null     float64
 2   Left Shoulder_y      73 non-null     float64
 3   Left Shoulder_conf   73 non-null     float64
 4   Right Shoulder_x     73 non-null     float64
 5   Right Shoulder_y     73 non-null     float64
 6   Right Shoulder_conf  73 non-null     float64
 7   Left Elbow_x         73 non-null     float64
 8   Left Elbow_y         73 non-null     float64
 9   Left Elbow_conf      73 non-null     float64
 10  Right Elbow_x        73 non-null     float64
 11  Right Elbow_y        73 non-null     float64
 12  Right Elbow_conf     73 non-null     float64
 13  Left Wrist_x         73 non-null     float64
 14  Left Wrist_y         73 non-null     float64
 15  Left Wrist_conf      73 non-null     float

In [31]:
df.max()

frame                    59.000000
Left Shoulder_x        1866.227539
Left Shoulder_y        1050.137329
Left Shoulder_conf        0.988900
Right Shoulder_x       1858.493042
Right Shoulder_y       1040.435791
Right Shoulder_conf       0.962918
Left Elbow_x           1876.223267
Left Elbow_y           1080.000000
Left Elbow_conf           0.146377
Right Elbow_x          1868.676392
Right Elbow_y          1080.000000
Right Elbow_conf          0.018749
Left Wrist_x           1920.000000
Left Wrist_y           1064.460571
Left Wrist_conf           0.783444
Right Wrist_x          1904.329468
Right Wrist_y          1065.752563
Right Wrist_conf          0.415467
Left Hip_x             1879.369385
Left Hip_y             1080.000000
Left Hip_conf             0.003477
Right Hip_x            1883.385986
Right Hip_y            1080.000000
Right Hip_conf            0.005730
Left Knee_x            1856.340088
Left Knee_y            1080.000000
Left Knee_conf            0.004756
Right Knee_x        

In [32]:
df.head()

,frame,Left Shoulder_x,Left Shoulder_y,Left Shoulder_conf,Right Shoulder_x,Right Shoulder_y,Right Shoulder_conf,Left Elbow_x,Left Elbow_y,Left Elbow_conf,...,Left Knee_conf,Right Knee_x,Right Knee_y,Right Knee_conf,Left Ankle_x,Left Ankle_y,Left Ankle_conf,Right Ankle_x,Right Ankle_y,Right Ankle_conf
0,0,1424.110962,993.762634,0.905596,704.344666,1020.448792,0.913182,1518.191406,1071.431396,0.005509,...,0.001585,838.337830,1080.0,0.001660,1192.874146,1080.0,0.000352,940.914551,1080.0,0.000696
1,1,1438.859985,1014.715759,0.933951,697.710754,1028.207520,0.940646,1517.736938,1080.000000,0.002958,...,0.001362,828.336609,1080.0,0.001547,1180.128418,1080.0,0.000240,928.343689,1080.0,0.000516
2,2,1446.010254,1031.935547,0.923083,698.586304,1031.863037,0.936629,1518.396851,1080.000000,0.002013,...,0.001350,817.569214,1080.0,0.001726,1171.809814,1080.0,0.000203,922.086548,1080.0,0.000456
3,3,1439.775635,1035.010986,0.920916,702.842834,1031.151489,0.937211,1517.457397,1080.000000,0.001792,...,0.001236,817.365417,1080.0,0.001657,1167.926147,1080.0,0.000184,924.311829,1080.0,0.000417
4,4,1439.810303,1035.710205,0.920352,700.170227,1030.109497,0.935427,1519.770630,1080.000000,0.001756,...,0.001230,815.877380,1080.0,0.001672,1170.679321,1080.0,0.000182,924.766174,1080.0,0.000422


In [33]:
df = df.drop(columns='frame')
df.head()

,Left Shoulder_x,Left Shoulder_y,Left Shoulder_conf,Right Shoulder_x,Right Shoulder_y,Right Shoulder_conf,Left Elbow_x,Left Elbow_y,Left Elbow_conf,Right Elbow_x,...,Left Knee_conf,Right Knee_x,Right Knee_y,Right Knee_conf,Left Ankle_x,Left Ankle_y,Left Ankle_conf,Right Ankle_x,Right Ankle_y,Right Ankle_conf
0,1424.110962,993.762634,0.905596,704.344666,1020.448792,0.913182,1518.191406,1071.431396,0.005509,613.872437,...,0.001585,838.337830,1080.0,0.001660,1192.874146,1080.0,0.000352,940.914551,1080.0,0.000696
1,1438.859985,1014.715759,0.933951,697.710754,1028.207520,0.940646,1517.736938,1080.000000,0.002958,634.289734,...,0.001362,828.336609,1080.0,0.001547,1180.128418,1080.0,0.000240,928.343689,1080.0,0.000516
2,1446.010254,1031.935547,0.923083,698.586304,1031.863037,0.936629,1518.396851,1080.000000,0.002013,639.363098,...,0.001350,817.569214,1080.0,0.001726,1171.809814,1080.0,0.000203,922.086548,1080.0,0.000456
3,1439.775635,1035.010986,0.920916,702.842834,1031.151489,0.937211,1517.457397,1080.000000,0.001792,645.795166,...,0.001236,817.365417,1080.0,0.001657,1167.926147,1080.0,0.000184,924.311829,1080.0,0.000417
4,1439.810303,1035.710205,0.920352,700.170227,1030.109497,0.935427,1519.770630,1080.000000,0.001756,643.418335,...,0.001230,815.877380,1080.0,0.001672,1170.679321,1080.0,0.000182,924.766174,1080.0,0.000422


In [39]:
sequence = []
dataset = []
window_size = 30

for frame in df.values:
    sequence.append(frame)
    
    if(len(sequence) == window_size):
        dataset.append(sequence.copy())
        sequence.pop(0)

dataset

[[array([     1424.1,      993.76,      0.9056,      704.34,      1020.4,     0.91318,      1518.2,      1071.4,   0.0055086,      613.87,        1080,   0.0029548,      1390.7,      1015.1,    0.017344,      885.41,      990.43,   0.0089468,      1299.6,        1080,   0.0007901,       851.2,        1080,  0.00086494,
              1332.9,        1052,    0.001585,      838.34,        1080,   0.0016602,      1192.9,        1080,  0.00035154,      940.91,        1080,  0.00069577]),
  array([     1438.9,      1014.7,     0.93395,      697.71,      1028.2,     0.94065,      1517.7,        1080,   0.0029584,      634.29,        1080,   0.0011577,      1338.3,        1054,   0.0076307,      874.37,      1055.4,   0.0026101,      1297.6,        1080,   0.0006804,      843.83,        1080,  0.00078826,
              1327.9,        1080,   0.0013616,      828.34,        1080,   0.0015474,      1180.1,        1080,  0.00023961,      928.34,        1080,  0.00051602]),
  array([       1446,   

In [41]:
dataset = np.array(dataset)

dataset.shape

(44, 30, 36)

In [44]:
input_dim = dataset.shape[2]
d_model = 128

In [48]:
linear = nn.Linear(input_dim, d_model)
linear

Linear(in_features=36, out_features=128, bias=True)

In [54]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        self.pos_embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        # x: (batch, seq, d_model)
        batch, seq_len, _ = x.shape

        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        pos_emb = self.pos_embedding(positions)

        return x + pos_emb
    
pos_encoder = PositionalEncoding(d_model)
pos_encoder

PositionalEncoding(
  (pos_embedding): Embedding(500, 128)
)

In [64]:
encoder_layer = nn.TransformerEncoderLayer(
    d_model=d_model,
    nhead=8 #divisible by d_model
    )

transformer = nn.TransformerEncoder(
    encoder_layer=encoder_layer,
    num_layers=2
)

/var/folders/45/btfyl_bj1mdd17yqtfkj5p2c0000gn/T/ipykernel_28522/2449820491.py:6: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  transformer = nn.TransformerEncoder(


In [73]:
x = torch.tensor(dataset, dtype=torch.float32)
x = linear(x)
x = pos_encoder(x)
x = transformer(x)

In [68]:
x.shape

torch.Size([44, 30, 128])

In [70]:
x

tensor([[[-0.6674,  0.9903, -1.7830,  ...,  0.4935,  0.3304,  0.4588],
         [-0.6692,  0.6087, -1.4872,  ...,  0.6386,  0.4308,  0.1186],
         [-1.0472,  0.8578, -1.3488,  ...,  0.7363,  0.3474, -0.1516],
         ...,
         [-0.8618,  1.3283, -1.7695,  ...,  0.5972,  0.3569, -0.0573],
         [-0.8662,  0.5677, -1.6030,  ...,  0.5417,  0.4155,  0.3684],
         [-1.4750,  0.9802, -1.4960,  ...,  0.7611,  0.4496,  0.4854]],

        [[-0.9665,  0.9537, -1.4787,  ...,  0.7262,  0.5857,  0.1668],
         [-1.1377,  1.0775, -1.7416,  ...,  0.7211,  0.5957,  0.1165],
         [-0.9727,  0.7329, -1.7449,  ...,  0.5731,  0.1326, -0.1557],
         ...,
         [-0.8586,  0.9770, -1.1837,  ...,  0.5417,  0.4301,  0.1930],
         [-1.0459,  1.0994, -1.4923,  ...,  0.8677,  0.5534,  0.1483],
         [-0.9365,  0.4216, -1.4094,  ...,  0.5170,  0.4799,  0.3588]],

        [[-1.3017,  0.9174, -1.6958,  ...,  0.6406,  0.3909, -0.1303],
         [-0.7240,  0.9172, -1.2041,  ...,  0

In [74]:
x = x.mean(dim=1)

classifier = nn.Linear(128, 1)
x = classifier(x)
x = torch.sigmoid(x)
x

tensor([[0.4285],
        [0.4131],
        [0.4231],
        [0.4271],
        [0.4257],
        [0.4212],
        [0.4273],
        [0.4306],
        [0.4298],
        [0.4390],
        [0.4396],
        [0.4382],
        [0.4274],
        [0.4310],
        [0.4295],
        [0.4256],
        [0.4321],
        [0.4306],
        [0.4233],
        [0.4332],
        [0.4364],
        [0.4348],
        [0.4264],
        [0.4341],
        [0.4353],
        [0.4351],
        [0.4384],
        [0.4345],
        [0.4299],
        [0.4430],
        [0.4267],
        [0.4368],
        [0.4339],
        [0.4377],
        [0.4213],
        [0.4338],
        [0.4324],
        [0.4272],
        [0.4210],
        [0.4341],
        [0.4332],
        [0.4314],
        [0.4270],
        [0.4355]], grad_fn=<SigmoidBackward0>)

In [ ]:
"""
COMPACT INTO A SINGLE CLASS
"""
class PoseEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.linear = nn.Linear(dataset.shape[2], 128)

        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8
        )

        self.transformer = nn.TransformerEncoder(
            self.encoder_layer,
            num_layers=2
        )

    def forward(self, x):
      

        x = self.linear(x)          # (B, 30, 128)
        x = self.transformer(x)     # (B, 30, 128)
        x = x.mean(dim=1)           # (B, 128) → embedding

        return x

In [ ]:
model = PoseEncoder(x)
torch.save(model.state_dict(), 'training/sample.pth')

/var/folders/45/btfyl_bj1mdd17yqtfkj5p2c0000gn/T/ipykernel_28522/2044121963.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(


In [87]:
model.eval()

PoseEncoder(
  (linear): Linear(in_features=36, out_features=128, bias=True)
  (encoder_layer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
    )
    (linear1): Linear(in_features=128, out_features=2048, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=2048, out_features=128, bias=True)
    (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
    (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_fe

In [79]:
good_embedding = model(torch.tensor(dataset, dtype=torch.float32))
good_embedding

tensor([[-1.7157,  0.9026,  0.8816,  ...,  0.9921,  0.7776, -0.6644],
        [-1.7160,  0.9030,  0.8819,  ...,  0.9912,  0.7776, -0.6640],
        [-1.7263,  0.9109,  0.8720,  ...,  0.9754,  0.7802, -0.6553],
        ...,
        [-1.7958,  0.9371,  0.8533,  ...,  0.8884,  0.8118, -0.5859],
        [-1.7951,  0.9368,  0.8535,  ...,  0.8921,  0.8090, -0.5874],
        [-1.7959,  0.9358,  0.8511,  ...,  0.8954,  0.8086, -0.5888]], grad_fn=<MeanBackward1>)

In [80]:
input_embedding = model(torch.tensor(dataset, dtype=torch.float32))
input_embedding

tensor([[-1.7157,  0.9026,  0.8816,  ...,  0.9921,  0.7776, -0.6644],
        [-1.7160,  0.9030,  0.8819,  ...,  0.9912,  0.7776, -0.6640],
        [-1.7263,  0.9109,  0.8720,  ...,  0.9754,  0.7802, -0.6553],
        ...,
        [-1.7958,  0.9371,  0.8533,  ...,  0.8884,  0.8118, -0.5859],
        [-1.7951,  0.9368,  0.8535,  ...,  0.8921,  0.8090, -0.5874],
        [-1.7959,  0.9358,  0.8511,  ...,  0.8954,  0.8086, -0.5888]], grad_fn=<MeanBackward1>)

In [85]:
similarity = F.cosine_similarity(
    input_embedding,
    good_embedding
)

print("Similarity score:", similarity.mean().item()) #same dataset is used thats why score is 1.0

Similarity score: 1.0


Reusable components

In [39]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import os
from torch.utils.data import TensorDataset, DataLoader

In [40]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0, max_len,
            dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-torch.log(torch.tensor(10000.0)) / d_model)
        )

        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self,x):
        return x + self.pe[:,:x.size(1)]

import torch.nn.functional as F

class PoseAutoEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        # Encoder with projection layer and LayerNorm
        self.linear = nn.Linear(input_dim, 128)
        self.layernorm = nn.LayerNorm(128)
        self.position = PositionalEncoding(128)

        # Transformer Encoder with dropout = 0.2
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            dropout=0.2,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # Bottleneck (latent space of 64)
        self.bottleneck_encode = nn.Linear(128, 64)
        self.bottleneck_decode = nn.Linear(64, 128)

        # Decoder
        self.decoder = nn.Linear(128, input_dim)

    def forward(self, x):
        x = self.linear(x)          # (B, T, input_dim) -> (B, T, 128)
        x = self.layernorm(x)       # (B, T, 128)
        x = self.position(x)        # (B, T, 128)
        x = self.transformer(x)     # (B, T, 128)

        # Bottleneck representation
        x = F.relu(self.bottleneck_encode(x))  # (B, T, 64)
        x = F.relu(self.bottleneck_decode(x))  # (B, T, 128)

        reconstructed = self.decoder(x)   # (B, T, input_dim)

        return reconstructed

def resample_sequence(sequence, target_frames):
    """
    sequence: numpy array of shape (frames, features)
    target_frames: desired number of frames

    Returns:
        numpy array of shape (target_frames, features)
    """

    original_frames = sequence.shape[0]
    num_features = sequence.shape[1]

    # Original frame positions
    x_old = np.arange(original_frames)

    # New frame positions
    x_new = np.linspace(0, original_frames - 1, target_frames)

    # Output array
    resampled = np.zeros((target_frames, num_features), dtype=np.float32)

    # Interpolate each feature independently
    for i in range(num_features):
        resampled[:, i] = np.interp(
            x_new,
            x_old,
            sequence[:, i]
        )

    return resampled
    
def normalize_pose(df):

    # Shoulder center
    center_x = (df["Left Shoulder_x"] + df["Right Shoulder_x"]) / 2
    center_y = (df["Left Shoulder_y"] + df["Right Shoulder_y"]) / 2

    # Shoulder width per frame
    sw_per_frame = np.sqrt(
        (df["Left Shoulder_x"] - df["Right Shoulder_x"]) ** 2 +
        (df["Left Shoulder_y"] - df["Right Shoulder_y"]) ** 2
    )
    
    # Use median shoulder width to be robust against outliers/failures
    shoulder_width = np.median(sw_per_frame)
    if shoulder_width < 1e-6:
        shoulder_width = 1e-6

    # Normalize every keypoint
    for col in df.columns:

        if col.endswith("_x"):
            df[col] = (df[col] - center_x) / shoulder_width

        elif col.endswith("_y"):
            df[col] = (df[col] - center_y) / shoulder_width

    return df

def prepare(folder):

    dataset = []

    for file in sorted(os.listdir(folder)):
        if not file.endswith(".csv"):
            continue

        seq = pd.read_csv(os.path.join(folder, file))
        seq = seq.drop(columns=["frame"])
        
        # Remove confidence columns
        conf_cols = [c for c in seq.columns if "conf" in c.lower()]
        seq = seq.drop(columns=conf_cols)

        seq = normalize_pose(seq)
        seq = seq.to_numpy(dtype=np.float32)
        seq = resample_sequence(seq, target_frames=200)

        dataset.append(seq)

    dataset = np.array(dataset, dtype=np.float32)
    
    for seq in dataset:
        print(seq.shape)

    train, temp = train_test_split(
        dataset,
        test_size=0.30,
        random_state=42,
        shuffle=True
    )

    val, test = train_test_split(
        temp,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    input_dim = train.shape[2]

    return train, val, test, input_dim


def build_model(input_dim):

    return PoseAutoEncoder(input_dim)

def get_reconstruction_error(model, sequence):
    """
    Computes reconstruction error (MSE) for a sequence.
    """
    model.eval()
    with torch.no_grad():
        if isinstance(sequence, np.ndarray):
            sequence = torch.tensor(sequence, dtype=torch.float32)
        if len(sequence.shape) == 2:
            sequence = sequence.unsqueeze(0)
        device = next(model.parameters()).device
        sequence = sequence.to(device)
        output = model(sequence)
        error = torch.mean((output - sequence) ** 2)
        return error.item()

def compute_similarity_score(error, mean_val_loss, beta):
    """
    Computes the similarity score based on reconstruction error using calibrated exponential decay.
    """
    if error <= mean_val_loss:
        return 100.0
    score = 100.0 * np.exp(-beta * (error - mean_val_loss))
    return float(np.clip(score, 0.0, 100.0))

def train_model(model, train_data, val_data):
    best_val_loss = float("inf")

    if os.path.exists("training/checkpoints/side_arms_raise_v1.pth"):
        model_checkpoint = torch.load("training/checkpoints/side_arms_raise_v1.pth", map_location="cpu")
        model.load_state_dict(model_checkpoint["model"])
        best_val_loss = model_checkpoint.get("best_val_loss", float("inf"))

    train_data = torch.tensor(train_data, dtype=torch.float32)
    val_data = torch.tensor(val_data, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(train_data),
        batch_size=8,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(val_data),
        batch_size=8,
        shuffle=False
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    epochs = 100

    for epoch in range(epochs):
        
        model.train()

        train_loss = 0.0

        for (batch,) in train_loader:

            optimizer.zero_grad()

            output = model(batch)

            loss = criterion(output, batch)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()

        val_loss = 0.0

        with torch.no_grad():

            for (batch,) in val_loader:

                output = model(batch)

                loss = criterion(output, batch)

                val_loss += loss.item()

        val_loss /= len(val_loader)

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            # Calculate calibration parameters on combined train + val data
            cal_errors = []
            with torch.no_grad():
                for sample in train_data:
                    sample = sample.unsqueeze(0)
                    output = model(sample)
                    cal_errors.append(torch.mean((output - sample) ** 2).item())
                for sample in val_data:
                    sample = sample.unsqueeze(0)
                    output = model(sample)
                    cal_errors.append(torch.mean((output - sample) ** 2).item())
            
            mean_val_loss = float(np.mean(cal_errors))
            std_val_loss = float(np.std(cal_errors))
            if std_val_loss < 1e-4:
                std_val_loss = 1e-4
            
            k = 3.0
            beta = float(np.log(2.0) / (k * std_val_loss))

            torch.save({
                "model": model.state_dict(),
                "best_val_loss": best_val_loss,
                "mean_val_loss": mean_val_loss,
                "std_val_loss": std_val_loss,
                "beta": beta
            }, "training/checkpoints/side_arms_raise_v1.pth")
            
            print(f"Saved new best model with val loss: {best_val_loss:.6f} at epoch {epoch+1:03d}")
            print(f"Calibrated stats -> Mean: {mean_val_loss:.6f}, Std: {std_val_loss:.6f}, Beta: {beta:.4f}")

        print(
            f"Epoch {epoch+1:03d} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f}"
        )

    return model


In [41]:
train, val, test, input_dim = prepare("data/shoulder_exercise_1")

model = build_model(input_dim)

model = train_model(model, train, val)

(200, 8)
(200, 8)
(200, 8)
(200, 8)
Epoch 001 | Train Loss: 0.013570 | Val Loss: 0.105124
Epoch 002 | Train Loss: 0.061564 | Val Loss: 0.129914
Epoch 003 | Train Loss: 0.034525 | Val Loss: 0.090522
Epoch 004 | Train Loss: 0.033162 | Val Loss: 0.048126
Saved new best model with val loss: 0.027802 at epoch 005
Calibrated stats -> Mean: 0.016596, Std: 0.007986, Beta: 28.9331
Epoch 005 | Train Loss: 0.024212 | Val Loss: 0.027802
Saved new best model with val loss: 0.026487 at epoch 006
Calibrated stats -> Mean: 0.019251, Std: 0.005340, Beta: 43.2642
Epoch 006 | Train Loss: 0.017076 | Val Loss: 0.026487
Epoch 007 | Train Loss: 0.019136 | Val Loss: 0.033343
Epoch 008 | Train Loss: 0.022835 | Val Loss: 0.038462
Epoch 009 | Train Loss: 0.020245 | Val Loss: 0.041777
Epoch 010 | Train Loss: 0.014769 | Val Loss: 0.046400
Epoch 011 | Train Loss: 0.010505 | Val Loss: 0.051862
Epoch 012 | Train Loss: 0.010238 | Val Loss: 0.055969
Epoch 013 | Train Loss: 0.011664 | Val Loss: 0.056359
Epoch 014 | Trai

In [42]:
# Load the saved model and calibration parameters
checkpoint = torch.load("training/checkpoints/side_arms_raise_v1.pth", map_location="cpu")
model.load_state_dict(checkpoint["model"])
mean_val_loss = checkpoint.get("mean_val_loss", 0.0)
beta = checkpoint.get("beta", 1.0)

print(f"\nModel Loaded. Calibration Parameters -> Mean Val Loss: {mean_val_loss:.6f}, Beta: {beta:.4f}\n")

# Evaluate on test set
print("=== Evaluation on Test Set ===")
for idx, test_seq in enumerate(test):
    error = get_reconstruction_error(model, test_seq)
    score = compute_similarity_score(error, mean_val_loss, beta)
    print(f"Test Sequence {idx+1} | Reconstruction Error: {error:.6f} | Movement Similarity Score: {score:.2f}%")



Model Loaded. Calibration Parameters -> Mean Val Loss: 0.019251, Beta: 43.2642

=== Evaluation on Test Set ===
Test Sequence 1 | Reconstruction Error: 0.016886 | Movement Similarity Score: 100.00%
